# DATA EXPLORATION - VANGUARD A/B TEST
---



Digital Footprints: A detailed trace of client interactions online
**Question was: Would these changes encourage more clients to complete the process?**


IMPORT LIBRARIES

In [148]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

sns.set_style("whitegrid")


1. LOAD THE DATASETS 

In [149]:
data_path = "data/raw_data"
df_web_1 = pd.read_csv(f"{data_path}/df_final_web_data_pt_1.txt")
df_web_2 = pd.read_csv(f"{data_path}/df_final_web_data_pt_2.txt")
df_experiment = pd.read_csv(f"{data_path}/df_final_experiment_clients.txt")


## Phase 1 — EDA & Data Cleaning

DATASET: Web_1

In [150]:
df_web_1.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


In [151]:
df_web_1.shape

(343141, 5)

In [152]:
df_web_1["client_id"].nunique()

58391

DATASET: Web_2

In [153]:
df_web_2.head()

,client_id,visitor_id,visit_id,process_step,date_time
0,763412,601952081_10457207388,397475557_40440946728_419634,confirm,2017-06-06 08:56:00
1,6019349,442094451_91531546617,154620534_35331068705_522317,confirm,2017-06-01 11:59:27
2,6019349,442094451_91531546617,154620534_35331068705_522317,step_3,2017-06-01 11:58:48
3,6019349,442094451_91531546617,154620534_35331068705_522317,step_2,2017-06-01 11:58:08
4,6019349,442094451_91531546617,154620534_35331068705_522317,step_1,2017-06-01 11:57:58


In [154]:
df_web_2.shape

(412264, 5)

In [155]:
df_web_2["client_id"].nunique()

67430

In [156]:
# merging both databases
df_web_3 = pd.concat([df_web_1,df_web_2], axis = 0, ignore_index = True)
df_web_3

,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04
...,...,...,...,...,...
755400,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:46:10
755401,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:45:29
755402,9668240,388766751_9038881013,922267647_3096648104_968866,step_1,2017-05-24 18:44:51
755403,9668240,388766751_9038881013,922267647_3096648104_968866,start,2017-05-24 18:44:34


In [157]:
web = df_web_3.merge(df_experiment, on='client_id', how='inner')

In [158]:
web.head(10)

,client_id,visitor_id,visit_id,process_step,date_time,Variation
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,Test
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,Test
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,Test
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,Test
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,Test
5,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:17:15,Test
6,9988021,580560515_7732621733,781255054_21935453173_531117,step_1,2017-04-17 15:17:01,Test
7,9988021,580560515_7732621733,781255054_21935453173_531117,start,2017-04-17 15:16:22,Test
8,8320017,39393514_33118319366,960651974_70596002104_312201,confirm,2017-04-05 13:10:05,Test
9,8320017,39393514_33118319366,960651974_70596002104_312201,step_3,2017-04-05 13:09:43,Test


In [159]:
#renaming columns
web.columns = web.columns.str.lower().str.replace(' ', '_')

In [160]:
web.dtypes

client_id        int64
visitor_id      object
visit_id        object
process_step    object
date_time       object
variation       object
dtype: object

**Dealing with NULL VALUES**

In [161]:
# finding nulls
web.isna().sum()

client_id            0
visitor_id           0
visit_id             0
process_step         0
date_time            0
variation       128522
dtype: int64

In [162]:
web.dropna(subset=['variation'], inplace=True)

**Sorting Date_time Column**

In [163]:
web['date_time'] = pd.to_datetime(web['date_time'])

In [164]:
web = web.sort_values(by=['client_id', 'visit_id', 'date_time'])

In [165]:
web['next_time'] = web.groupby(['client_id', 'visit_id'])['date_time'].shift(-1)

In [166]:
web['sec_spent'] = (web['next_time'] - web['date_time']).dt.total_seconds()

In [167]:
web.head(10)

,client_id,visitor_id,visit_id,process_step,date_time,variation,next_time,sec_spent
70803,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,2017-04-15 12:58:03,7.0
70802,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,2017-04-15 12:58:35,32.0
70801,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,2017-04-15 13:00:14,99.0
70800,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,2017-04-15 13:00:34,20.0
70799,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,NaT,NaN
135918,647,66758770_53988066587,40369564_40101682850_311847,start,2017-04-12 15:41:28,Test,2017-04-12 15:41:35,7.0
135917,647,66758770_53988066587,40369564_40101682850_311847,step_1,2017-04-12 15:41:35,Test,2017-04-12 15:41:53,18.0
135916,647,66758770_53988066587,40369564_40101682850_311847,step_2,2017-04-12 15:41:53,Test,2017-04-12 15:45:02,189.0
135915,647,66758770_53988066587,40369564_40101682850_311847,step_3,2017-04-12 15:45:02,Test,2017-04-12 15:47:45,163.0
135912,647,66758770_53988066587,40369564_40101682850_311847,confirm,2017-04-12 15:47:45,Test,NaT,NaN


In [168]:
web['process_order'] = web.groupby(['client_id', 'visit_id'])['process_step'].shift(-1)

In [169]:
web.head()

,client_id,visitor_id,visit_id,process_step,date_time,variation,next_time,sec_spent,process_order
70803,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,2017-04-15 12:58:03,7.0,step_1
70802,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,2017-04-15 12:58:35,32.0,step_2
70801,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,2017-04-15 13:00:14,99.0,step_3
70800,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,2017-04-15 13:00:34,20.0,confirm
70799,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,NaT,NaN,NaN


In [170]:
web.head()

,client_id,visitor_id,visit_id,process_step,date_time,variation,next_time,sec_spent,process_order
70803,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,2017-04-15 12:58:03,7.0,step_1
70802,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,2017-04-15 12:58:35,32.0,step_2
70801,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,2017-04-15 13:00:14,99.0,step_3
70800,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,2017-04-15 13:00:34,20.0,confirm
70799,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,NaT,NaN,NaN


In [171]:
correct_order = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

In [172]:
expected_next = {
    'start': 'step_1',
    'step_1': 'step_2',
    'step_2': 'step_3',
    'step_3': 'confirm',
    'confirm': None
}

In [173]:
def check_row(row):
    current = row['process_step']
    actual_next = row['process_order']
    expected = expected_next.get(current)
    
    # repeated
    if current == actual_next:
        return "repeated"
    
    # correct
    if expected == actual_next:
        return "correct"
    
    # handle last step (confirm → NaN)
    if expected is None and pd.isna(actual_next):
        return "correct"
    
    return "error"

In [174]:
web['process_status'] = web.apply(check_row, axis=1)

In [175]:
web.head()

,client_id,visitor_id,visit_id,process_step,date_time,variation,next_time,sec_spent,process_order,process_status
70803,555,402506806_56087378777,637149525_38041617439_716659,start,2017-04-15 12:57:56,Test,2017-04-15 12:58:03,7.0,step_1,correct
70802,555,402506806_56087378777,637149525_38041617439_716659,step_1,2017-04-15 12:58:03,Test,2017-04-15 12:58:35,32.0,step_2,correct
70801,555,402506806_56087378777,637149525_38041617439_716659,step_2,2017-04-15 12:58:35,Test,2017-04-15 13:00:14,99.0,step_3,correct
70800,555,402506806_56087378777,637149525_38041617439_716659,step_3,2017-04-15 13:00:14,Test,2017-04-15 13:00:34,20.0,confirm,correct
70799,555,402506806_56087378777,637149525_38041617439_716659,confirm,2017-04-15 13:00:34,Test,NaT,NaN,NaN,correct


In [176]:
web = web.reindex(columns=['client_id', 'visitor_id', 'visit_id', 'process_step','process_order', 'process_status', 'date_time', 'next_time', 'sec_spent', 'variation'])

In [177]:
web.head(50)

,client_id,visitor_id,visit_id,process_step,process_order,process_status,date_time,next_time,sec_spent,variation
70803,555,402506806_56087378777,637149525_38041617439_716659,start,step_1,correct,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test
70802,555,402506806_56087378777,637149525_38041617439_716659,step_1,step_2,correct,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test
70801,555,402506806_56087378777,637149525_38041617439_716659,step_2,step_3,correct,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test
70800,555,402506806_56087378777,637149525_38041617439_716659,step_3,confirm,correct,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test
70799,555,402506806_56087378777,637149525_38041617439_716659,confirm,NaN,correct,2017-04-15 13:00:34,NaT,NaN,Test
135918,647,66758770_53988066587,40369564_40101682850_311847,start,step_1,correct,2017-04-12 15:41:28,2017-04-12 15:41:35,7.0,Test
135917,647,66758770_53988066587,40369564_40101682850_311847,step_1,step_2,correct,2017-04-12 15:41:35,2017-04-12 15:41:53,18.0,Test
135916,647,66758770_53988066587,40369564_40101682850_311847,step_2,step_3,correct,2017-04-12 15:41:53,2017-04-12 15:45:02,189.0,Test
135915,647,66758770_53988066587,40369564_40101682850_311847,step_3,confirm,correct,2017-04-12 15:45:02,2017-04-12 15:47:45,163.0,Test
135912,647,66758770_53988066587,40369564_40101682850_311847,confirm,NaN,correct,2017-04-12 15:47:45,NaT,NaN,Test


In [178]:
web.head(20)

,client_id,visitor_id,visit_id,process_step,process_order,process_status,date_time,next_time,sec_spent,variation
70803,555,402506806_56087378777,637149525_38041617439_716659,start,step_1,correct,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test
70802,555,402506806_56087378777,637149525_38041617439_716659,step_1,step_2,correct,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test
70801,555,402506806_56087378777,637149525_38041617439_716659,step_2,step_3,correct,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test
70800,555,402506806_56087378777,637149525_38041617439_716659,step_3,confirm,correct,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test
70799,555,402506806_56087378777,637149525_38041617439_716659,confirm,NaN,correct,2017-04-15 13:00:34,NaT,NaN,Test
135918,647,66758770_53988066587,40369564_40101682850_311847,start,step_1,correct,2017-04-12 15:41:28,2017-04-12 15:41:35,7.0,Test
135917,647,66758770_53988066587,40369564_40101682850_311847,step_1,step_2,correct,2017-04-12 15:41:35,2017-04-12 15:41:53,18.0,Test
135916,647,66758770_53988066587,40369564_40101682850_311847,step_2,step_3,correct,2017-04-12 15:41:53,2017-04-12 15:45:02,189.0,Test
135915,647,66758770_53988066587,40369564_40101682850_311847,step_3,confirm,correct,2017-04-12 15:45:02,2017-04-12 15:47:45,163.0,Test
135912,647,66758770_53988066587,40369564_40101682850_311847,confirm,NaN,correct,2017-04-12 15:47:45,NaT,NaN,Test


In [179]:
web.head(20)

,client_id,visitor_id,visit_id,process_step,process_order,process_status,date_time,next_time,sec_spent,variation
70803,555,402506806_56087378777,637149525_38041617439_716659,start,step_1,correct,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test
70802,555,402506806_56087378777,637149525_38041617439_716659,step_1,step_2,correct,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test
70801,555,402506806_56087378777,637149525_38041617439_716659,step_2,step_3,correct,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test
70800,555,402506806_56087378777,637149525_38041617439_716659,step_3,confirm,correct,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test
70799,555,402506806_56087378777,637149525_38041617439_716659,confirm,NaN,correct,2017-04-15 13:00:34,NaT,NaN,Test
135918,647,66758770_53988066587,40369564_40101682850_311847,start,step_1,correct,2017-04-12 15:41:28,2017-04-12 15:41:35,7.0,Test
135917,647,66758770_53988066587,40369564_40101682850_311847,step_1,step_2,correct,2017-04-12 15:41:35,2017-04-12 15:41:53,18.0,Test
135916,647,66758770_53988066587,40369564_40101682850_311847,step_2,step_3,correct,2017-04-12 15:41:53,2017-04-12 15:45:02,189.0,Test
135915,647,66758770_53988066587,40369564_40101682850_311847,step_3,confirm,correct,2017-04-12 15:45:02,2017-04-12 15:47:45,163.0,Test
135912,647,66758770_53988066587,40369564_40101682850_311847,confirm,NaN,correct,2017-04-12 15:47:45,NaT,NaN,Test


**Dealing with Duplicates**

In [180]:
# duplicates sum
df_web_3.duplicated().sum()

np.int64(10764)

## Descriptive Analysis 

In [181]:
web["client_id"].nunique()

50500

In [182]:
web["visitor_id"].nunique()
	

56011

In [183]:
web["process_step"].nunique()

5

In [184]:
web["sec_spent"].value_counts()

sec_spent
5.0       6337
4.0       6312
6.0       6207
7.0       5910
8.0       5650
          ... 
1694.0       1
1286.0       1
1240.0       1
2178.0       1
1456.0       1
Name: count, Length: 1864, dtype: int64

In [185]:
web["sec_spent"].describe().round(2)

count    251862.00
mean         82.51
std         212.57
min           0.00
25%          12.00
50%          35.00
75%          81.00
max       40235.00
Name: sec_spent, dtype: float64

In [186]:
web["process_status"].value_counts()

process_status
correct     224975
error        59631
repeated     36703
Name: count, dtype: int64

In [187]:
# 1. Ensure the data is sorted chronologically so the "first" row is truly the earliest
web = web.sort_values(by=['client_id', 'date_time'])

# 2. Filter on the fly without adding new columns
web_first_session = web[
    web['visit_id'] == web['client_id'].map(web.groupby('client_id')['visit_id'].first())
]

# Check your results
print(f"Original rows: {web.shape[0]}")
print(f"Rows after keeping only the first session: {web_first_session.shape[0]}")
print(f"Unique clients remaining: {web_first_session['client_id'].nunique()}")

Original rows: 321309
Rows after keeping only the first session: 248917
Unique clients remaining: 50500


In [188]:
web.shape

(321309, 10)

In [189]:
# Check if any rows are completely identical across client, step, and time
glitch_duplicates = web.duplicated(subset=['client_id', 'process_step', 'date_time']).sum()

print(f"Number of exact duplicate rows: {glitch_duplicates}")

# If the number is greater than 0, you can drop them like this:
if glitch_duplicates > 0:
    web = web.drop_duplicates(subset=['client_id', 'process_step', 'date_time'], keep='first')
    print("Glitch duplicates removed!")

Number of exact duplicate rows: 4074
Glitch duplicates removed!


In [190]:
web.head(50)

,client_id,visitor_id,visit_id,process_step,process_order,process_status,date_time,next_time,sec_spent,variation
70803,555,402506806_56087378777,637149525_38041617439_716659,start,step_1,correct,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test
70802,555,402506806_56087378777,637149525_38041617439_716659,step_1,step_2,correct,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test
70801,555,402506806_56087378777,637149525_38041617439_716659,step_2,step_3,correct,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test
70800,555,402506806_56087378777,637149525_38041617439_716659,step_3,confirm,correct,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test
70799,555,402506806_56087378777,637149525_38041617439_716659,confirm,NaN,correct,2017-04-15 13:00:34,NaT,NaN,Test
135918,647,66758770_53988066587,40369564_40101682850_311847,start,step_1,correct,2017-04-12 15:41:28,2017-04-12 15:41:35,7.0,Test
135917,647,66758770_53988066587,40369564_40101682850_311847,step_1,step_2,correct,2017-04-12 15:41:35,2017-04-12 15:41:53,18.0,Test
135916,647,66758770_53988066587,40369564_40101682850_311847,step_2,step_3,correct,2017-04-12 15:41:53,2017-04-12 15:45:02,189.0,Test
135915,647,66758770_53988066587,40369564_40101682850_311847,step_3,confirm,correct,2017-04-12 15:45:02,2017-04-12 15:47:45,163.0,Test
135912,647,66758770_53988066587,40369564_40101682850_311847,confirm,NaN,correct,2017-04-12 15:47:45,NaT,NaN,Test


In [191]:
web['process_status'].value_counts()

process_status
correct     223175
error        58777
repeated     35283
Name: count, dtype: int64

In [192]:
web.groupby('client_id')['visitor_id'].value_counts().sort_values(ascending=False)

client_id  visitor_id           
465007     819447509_47703321203    72
9638063    409434116_1011445691     63
2313292    528332823_62425640679    61
7597144    181494464_67394447335    58
5902055    806675212_20364284341    57
                                    ..
3784014    848657790_81641486585     1
3786978    114922688_59757740576     1
3788952    932735920_17501095763     1
           577750751_12633748227     1
8955423    955788863_12725740675     1
Name: count, Length: 56373, dtype: int64

In [193]:
web.groupby('client_id')['visit_id'].value_counts().sort_values(ascending=False)

client_id  visit_id                    
2313292    712824876_8175482950_365042     61
7795550    428529357_6959155752_124163     42
2138772    428919026_83099642366_340343    36
5582954    834703874_81652602361_748606    35
6686840    699879519_24719899358_572278    34
                                           ..
9994633    278477876_89906801725_644188     1
9995265    702318250_2028572774_273570      1
1109486    630738980_34351985589_304538     1
1112135    341041066_58946721190_432653     1
1112209    836093568_54737880252_384732     1
Name: count, Length: 69447, dtype: int64

In [194]:
# 1. Filter the dataframe for that exact client ID
busy_client = web[web['client_id'] == 2313292]

# 2. Make sure it's sorted chronologically so you can read it like a timeline
busy_client = busy_client.sort_values(by='date_time')

# 3. Print the results, forcing Pandas to show every single row
with pd.option_context('display.max_rows', None):
    print(busy_client[['client_id', 'process_step', 'date_time', 'process_status']])

        client_id process_step           date_time process_status
209785    2313292        start 2017-04-12 17:13:27        correct
209784    2313292       step_1 2017-04-12 17:13:56        correct
209783    2313292       step_2 2017-04-12 17:14:11          error
209782    2313292       step_1 2017-04-12 17:14:51        correct
209781    2313292       step_2 2017-04-12 17:14:59          error
209780    2313292       step_1 2017-04-12 17:15:02        correct
209779    2313292       step_2 2017-04-12 17:15:17        correct
209778    2313292       step_3 2017-04-12 17:15:44          error
209777    2313292       step_2 2017-04-12 17:17:08        correct
209776    2313292       step_3 2017-04-12 17:17:20       repeated
209775    2313292       step_3 2017-04-12 17:17:41          error
209774    2313292       step_2 2017-04-12 17:18:10          error
209773    2313292       step_1 2017-04-12 17:18:13        correct
209772    2313292       step_2 2017-04-12 17:18:18        correct
209771    

## Exporting Database

In [195]:

# exporting database
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

# Save the filtered dataframe
web.to_csv(f"{output_dir}/web.csv", index=False)

print("Exported:")
for f in os.listdir(output_dir):
    path = os.path.join(output_dir, f)
    size = os.path.getsize(path)
    print(f"  {f} ({size:,} bytes)")

Exported:
  df_exde.csv (3,302,143 bytes)
  df_web_3.csv (81,420,206 bytes)
  vanguard_client_summary.csv (1,354,845 bytes)
  vanguard_hypo.csv (3,884,978 bytes)
  vanguard_web_wrangled.csv (35,857,674 bytes)
  web.csv (39,957,052 bytes)
